# LC 56 — Merge Intervals
**Day 51 | Theme: Intervals | Difficulty: Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Sort intervals by start time.
Then walk forward: if the next interval's start is within
the current interval's end, they overlap — extend the end.
Otherwise, seal the current interval and start a new one.
</div>

## Official Problem Statement

Given an array of `intervals` where `intervals[i] = [starti, endi]`,
merge all overlapping intervals, and return an array of the
non-overlapping intervals that cover all the intervals in the input.

**Constraints:**
- `1 <= intervals.length <= 10^4`
- `intervals[i].length == 2`
- `0 <= starti <= endi <= 10^4`

## What This Is Actually Asking

You have a list of time ranges that may overlap each other.
Your job is to collapse any overlapping ranges into one.
Two intervals overlap when one starts before the other ends.
The result is a clean, non-overlapping list of intervals.
Order of the output follows sorted start times.

## Walk Through an Example by Hand

Input: `[[1,3],[2,6],[8,10],[15,18]]`

**Step 1 — Sort by start (already sorted here):**
```
[1,3], [2,6], [8,10], [15,18]
```

**Step 2 — Initialize:** `cur = [1, 3]`

**Step 3 — Process [2,6]:**
- 2 <= 3 → overlap! cur[1] = max(3,6) = 6
- cur = [1, 6]

**Step 4 — Process [8,10]:**
- 8 > 6 → no overlap. Append [1,6]. cur = [8,10]

**Step 5 — Process [15,18]:**
- 15 > 10 → no overlap. Append [8,10]. cur = [15,18]

**Step 6 — End of loop:** Append last cur = [15,18]

**Output:** `[[1,6],[8,10],[15,18]]`

## The Picture

```
Timeline (axis = time units 0..18)

Input:
  [1,3]   |==|
  [2,6]     |====|
  [8,10]            |==|
  [15,18]                   |===|

  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18

Overlap check:
  [1,3] and [2,6] → 2 <= 3 → MERGE → [1,6]
  [1,6] and [8,10] → 8 > 6 → SEAL [1,6], start [8,10]
  [8,10] and [15,18] → 15 > 10 → SEAL [8,10], start [15,18]
  End → SEAL [15,18]

Output:
  [1,6]   |========|
  [8,10]            |==|
  [15,18]                   |===|

  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18
```

## When To Use This Pattern

- When you see "overlapping intervals", think **sort-then-sweep**.
- When intervals need collapsing, think **track current, extend
  or seal**.
- When asked about coverage or gaps, think **sorted interval
  merge**.
- When a problem has start/end pairs in any order, think **sort
  by start first**.
- When merging ranges in scheduling/calendars, think **this
  pattern exactly**.

## The Approach

Sort all intervals by their start value so overlapping ones are
adjacent. Initialize `cur` to the first interval. Walk through
remaining intervals: if the next interval starts within `cur`,
extend `cur`'s end to the max of both ends. Otherwise, seal
`cur` into the result and make the next interval the new `cur`.
After the loop, always append the last `cur`.

In [ ]:
from typing import List

In [ ]:
def test_harness(func):
    """Run test cases for merge_intervals."""
    def norm(intervals):
        """Sort intervals for stable comparison."""
        return sorted([sorted(iv) for iv in intervals])

    cases = [
        # (input_intervals, expected)
        ([[1,3],[2,6],[8,10],[15,18]],
         [[1,6],[8,10],[15,18]]),
        ([[1,4],[4,5]],
         [[1,5]]),            # touching edges merge
        ([[1,4],[2,3]],
         [[1,4]]),            # one contained in other
        ([[1,4]],
         [[1,4]]),            # single interval
        ([[1,2],[3,4],[5,6]],
         [[1,2],[3,4],[5,6]]),# no overlaps
        ([[1,10],[2,3],[4,5]],
         [[1,10]]),           # multiple contained
        ([[2,3],[1,4]],
         [[1,4]]),            # unsorted input
    ]

    passed = 0
    for i, (intervals, expected) in enumerate(cases):
        result = func([iv[:] for iv in intervals])
        if norm(result) == norm(expected):
            print(f"  Case {i+1}: PASSED")
            passed += 1
        else:
            print(f"  Case {i+1}: FAILED")
            print(f"    Input:    {intervals}")
            print(f"    Expected: {expected}")
            print(f"    Got:      {result}")

    total = len(cases)
    print(f"\nResult: {passed}/{total} passed")
    if passed == total:
        print("All tests PASSED!")
    else:
        print(f"{total - passed} test(s) FAILED.")

In [ ]:
def merge(intervals: List[List[int]]) -> List[List[int]]:
    """
    Merge all overlapping intervals.

    Strategy:
        1. Sort by start time.
        2. Track current interval (cur).
        3. If next starts within cur, extend cur's end.
        4. Otherwise seal cur, move to next.
        5. Append the last cur after loop.

    Args:
        intervals: List of [start, end] pairs (unsorted ok).

    Returns:
        Merged non-overlapping list of intervals.

    Time:  O(n log n) — sort dominates
    Space: O(n) — output list
    """
    # Debug: show sorted input
    intervals.sort(key=lambda x: x[0])
    print(f"[DEBUG] sorted intervals: {intervals}")

    # TODO: initialize cur and result
    # TODO: iterate and merge or seal
    # TODO: append last cur
    pass


# Quick smoke test
sample = [[1,3],[2,6],[8,10],[15,18]]
print(f"[DEBUG] merge({sample}) = {merge(sample)}")

In [ ]:
# Uncomment and run when solution is ready
# test_harness(merge)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute force (check all pairs) | O(n²) | O(n) | Re-scan after each merge |
| Sort + sweep (optimal) | O(n log n) | O(n) | Sort dominates; single pass |

The bottleneck is the sort. The sweep itself is O(n) — each
interval is visited exactly once.

## Real World Connection

At **Citi**, trading windows and blackout periods are modeled as
time intervals — merging overlapping blackout windows ensures
compliance reporting shows a clean set of restricted periods.
On **AWS**, resource reservation systems (EC2 reserved capacity,
Redshift maintenance windows) merge overlapping schedules to
compute true downtime.
In **data engineering**, log file partitions from distributed
writers often have overlapping time ranges — merging them is a
prerequisite for deduplication and accurate time-series joins.
This pattern directly applies to calendar APIs, Gantt chart
libraries, and any pipeline that aggregates event streams.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra